# Team MLB · Random Forest

| 수행 내용 | 기본 실행 |
| --- | --- |
| Closed-world 95개 사이트 식별 | 실행 |
| 튜닝 전·후 비교 | 실행 |
| Open-world 이진 / 96클래스 다중 분류 | 둘 다 실행 |
| 균형 가중치 비교 | 실행 |
| 특징 중요도 / 사이트별 오류 분석 | 실행 |
| 그룹별 제거: Closed-world / 이진 | 둘 다 실행 |
| CV 결과·평가·예측 CSV 저장 | 실행 |

**탐색은 5개 후보 × 5-fold = 25회입니다.** 24개 전체 조합을 훑던 120회에서 줄였습니다.

| 후보 | 트리 수 | 최대 깊이 | 분기 특징 수 | 최소 리프 표본 수 |
| --- | --- | --- | --- | --- |
| 기준 | 100 | 10 | sqrt | 1 |
| 트리 수 변경 | 500 | 10 | sqrt | 1 |
| 깊이 변경 | 100 | 20 | sqrt | 1 |
| 분기 특징 수 변경 | 100 | 10 | 0.5 | 1 |
| 리프 표본 수 변경 | 100 | 10 | sqrt | 3 |

- 트리 수는 100~500 범위의 양 끝을 비교합니다. 300은 초기 후보에서 생략합니다.
- 튜닝 모델의 깊이를 10/20으로 제한합니다. 기본값 비교용 RF만 scikit-learn 기본 깊이(None)를 유지합니다.
- 각 항목은 탐색하지만 모든 조합과 상호작용을 훑지 않습니다. 전체 탐색과 같은 최적 성능을 보장하지 않습니다.
- 5-fold, 공유 데이터 분할, 평가 지표는 유지하고 표준화하지 않습니다.
- 그룹 제거도 선택한 모델과 동일한 설정 및 5-fold로 비교합니다. 그룹 제거 실험은 시간이 추가로 필요합니다.

**입력 범위:** 요청에 따라 누적 흐름 50개·trim0만 사용합니다. 두 입력 CSV만 필요합니다.
100개 지점 비교와 trim10/20 비교는 이 실행 범위에서 제외됩니다. 원본 pkl 보조 분석은 기본 생략입니다.

**실행:** `FEATURE_DIR`을 두 CSV의 폴더로 맞춘 뒤 위에서부터 실행하세요.
결과는 `results/random_forest_50points_required`에 저장합니다. 기존 결과와 분리합니다.

**중단 후 재실행:**
- 성공적으로 끝난 CV fold의 점수를 즉시 저장합니다. 같은 입력·설정으로 재실행하면 해당 fold는 재사용합니다.
- 데이터·특징·정답·분할·seed·fold 수·모델 설정·라이브러리 버전을 검사해 다른 실험의 점수를 섞지 않습니다.
- fold 내부에서 중단한 트리는 저장하지 않습니다. 현재 진행 중인 fold와 최종 재학습은 필요하면 처음부터 실행합니다.
- 예전 버전에서 이미 진행한 탐색은 이어받지 못합니다. 이 버전으로 완료한 fold부터 적용됩니다.
- 각 실험의 테스트 평가까지 끝나면 결과·예측·특징 중요도도 저장하여 재사용합니다.

**간결한 로그:** 시작 1줄, 후보별 완료 1줄, 5분 간격 진행(실제 완료 fold 수), 재학습·최종 결과만 표시합니다.
개별 fold 로그와 30초 CPU 상태 로그는 출력하지 않습니다. 상세 CV 표는 CSV로 저장합니다.

## 0. 설정 및 드라이브 연결

In [1]:
FEATURE_DIR = "/content/drive/MyDrive/MLB-Project/feature"
FEATURE_DIRS = {50: FEATURE_DIR}
RESULT_ROOT = "/content/drive/MyDrive/MLB-Project/results"
RESULT_DIR = RESULT_ROOT + "/random_forest_50points_required"
RAW_DIR = "/content/drive/MyDrive/MLB-Project/data"
SPLIT_PATH = RESULT_ROOT + "/split_seed42.csv"  # LR/RF 공통 파일
UNMON_NAME = "unmon10k"
CUMUL_POINTS_TO_COMPARE = [50]  # 누적 흐름 50개 지점만 사용
TRIM = 0
TRIMS_TO_COMPARE = [0]  # 현재 제공된 trim0 파일만 필요
SEED = 42
TEST_SIZE = 0.2
N_FOLDS = 5
N_JOBS = -1
SEARCH_N_JOBS = 1
RUN_ABLATION = True  # 팀 대표 모델만 True로 설정 가능. Closed-world/이진 그룹 제거 모두 제어
RUN_RAW_DIRECTION = False  # 원본 pkl 보조 분석은 필요할 때만 켜세요
# 소수 후보로 모든 탐색 항목을 포함합니다. 5개 후보 × 5-fold = 25회.
# 기준 설정에서 한 항목만 변경합니다. 모든 조합을 탐색하지는 않습니다.
RF_PARAM_GRID = [
    {"n_estimators": [100], "max_depth": [10], "max_features": ["sqrt"], "min_samples_leaf": [1]},
    {"n_estimators": [500], "max_depth": [10], "max_features": ["sqrt"], "min_samples_leaf": [1]},
    {"n_estimators": [100], "max_depth": [20], "max_features": ["sqrt"], "min_samples_leaf": [1]},
    {"n_estimators": [100], "max_depth": [10], "max_features": [0.5],    "min_samples_leaf": [1]},
    {"n_estimators": [100], "max_depth": [10], "max_features": ["sqrt"], "min_samples_leaf": [3]},
]
USE_FOLD_CACHE = True  # 완료한 fold 점수를 저장해 동일 조건 재실행에 재사용
SHOW_CV_TABLE = False  # 상세 CV 표는 CSV로 저장하고 화면에는 필요할 때만 표시
USE_COMPLETED_CACHE = True  # 완료한 동일 실험의 평가 결과를 저장하고 재사용합니다.

LOG_INTERVAL_SECONDS = 300  # 후보 완료 로그 + 5분 간격 상태만 출력

In [2]:
import os, sys, time, json, pickle, warnings, threading
import sklearn
from joblib import hash as joblib_hash
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, train_test_split, ParameterGrid
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60)

Mounted at /content/drive


## 1. CSV 점검 및 데이터 로드

누적 흐름 g6 특징 50개와 다른 특징 60개를 사용합니다. 정답 및 정보 열 6개는 모델 입력에서 제외합니다. NaN/inf가 있으면 특징 데이터를 점검하도록 중단합니다.

In [3]:
INFO_COLS = ["record_id", "dataset", "page_id", "sample_idx", "label", "trim"]
GROUP_SIZES = {"g1": 3, "g2": 17, "g3": 6, "g4": 3, "g5": 31}
MANIFEST = []

def feature_path(kind, trim, points):
    base = Path(FEATURE_DIRS[points])
    tagged = f"features_{kind}_trim{trim}_cumul{points}.csv"
    candidates = list(dict.fromkeys([
        base / tagged,
        base / f"features_{kind}_trim{trim}.csv",
        Path(FEATURE_DIR) / tagged,
    ]))
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"누적 흐름 {points}개 / {kind} / trim{trim} CSV가 없습니다.\n"
        + "확인한 경로:\n" + "\n".join(str(p) for p in candidates)
        + "\nFEATURE_DIR에 두 trim0 CSV가 있는지 확인하세요."
    )

def feature_columns(columns, points):
    feats = [c for c in columns if any(c.startswith(g + "_") for g in ["g1","g2","g3","g4","g5","g6"])]
    expected = {**GROUP_SIZES, "g6": points}
    actual = {g: sum(c.startswith(g + "_") for c in feats) for g in expected}
    if actual != expected:
        raise ValueError(f"특징 구성 불일치: 실제 {actual}, 필요 {expected}. "
                         "다른 지점 수의 CSV를 잘못 선택했는지 확인하세요.")
    return feats

# 오래 걸리는 학습 전에 모든 필수 CSV의 경로와 헤더를 검사합니다.
assert CUMUL_POINTS_TO_COMPARE and len(set(CUMUL_POINTS_TO_COMPARE)) == len(CUMUL_POINTS_TO_COMPARE)
assert CUMUL_POINTS_TO_COMPARE == [50], "이 노트북은 누적 흐름 50개 지점만 사용합니다."
for points in CUMUL_POINTS_TO_COMPARE:
    for trim in sorted(set([TRIM] + TRIMS_TO_COMPARE)):
        for kind in ["mon", UNMON_NAME]:
            path = feature_path(kind, trim, points)
            cols = pd.read_csv(path, nrows=0).columns.tolist()
            if cols[:6] != INFO_COLS:
                raise ValueError(f"정보 열 순서 불일치: {path}")
            feats = feature_columns(cols, points)
            MANIFEST.append(dict(cumul_points=points, trim=trim, dataset=kind,
                                 n_features=len(feats), source_csv=str(path)))
display(pd.DataFrame(MANIFEST))

def load_features(kind, trim, points):
    path = feature_path(kind, trim, points)
    df = pd.read_csv(path, dtype={"record_id": str})
    feats = feature_columns(df.columns, points)
    if df.record_id.isna().any() or df.record_id.duplicated().any():
        raise ValueError(f"record_id 누락/중복: {path}")
    if not df.dataset.eq(kind).all() or not df.trim.eq(trim).all():
        raise ValueError(f"dataset/trim 값과 파일 조건 불일치: {path}")
    if df.label.isna().any() or (kind == "mon" and not df.label.ge(0).all()) or (kind != "mon" and not df.label.eq(-1).all()):
        raise ValueError(f"정답 라벨 확인 필요: {path}")
    vals = df[feats].to_numpy(dtype=float)
    if not np.isfinite(vals).all():
        raise ValueError(f"NaN/inf 발견: {path}. 특징 추출 데이터 점검 후 다시 실행하세요.")
    print(f"[{points}지점/trim{trim}] {path.name}: {df.shape}")
    return df, feats

BASE_POINTS = CUMUL_POINTS_TO_COMPARE[0]
mon, FEATS = load_features("mon", TRIM, BASE_POINTS)
unmon, ufeats = load_features(UNMON_NAME, TRIM, BASE_POINTS)
assert FEATS == ufeats, "mon/unmon 특징 열과 순서가 달라요."
if set(mon.record_id) & set(unmon.record_id):
    raise ValueError("mon/unmon의 record_id는 서로 겹치면 안 됩니다.")

,cumul_points,trim,dataset,n_features,source_csv
0,50,0,mon,110,/content/drive/MyDrive/MLB-Project/feature/fea...
1,50,0,unmon10k,110,/content/drive/MyDrive/MLB-Project/feature/fea...


[50지점/trim0] features_mon_trim0.csv: (19000, 116)
[50지점/trim0] features_unmon10k_trim0.csv: (10000, 116)


## 2. 공유 학습/테스트 분할

mon은 하위 페이지별 비율을 유지하고 unmon은 기록 단위로 무작위 분할합니다. 기존 LR/RF와 같은 `split_seed42.csv`를 사용합니다. 테스트 기록은 하이퍼파라미터 선택에 사용하지 않습니다.

In [4]:
all_ids = set(mon.record_id) | set(unmon.record_id)
if Path(SPLIT_PATH).is_file():
    split = pd.read_csv(SPLIT_PATH, dtype={"record_id": str})
    print("기존 공유 분할 사용:", SPLIT_PATH)
else:
    tr_m, te_m = train_test_split(mon.record_id, test_size=TEST_SIZE, random_state=SEED, stratify=mon.page_id)
    tr_u, te_u = train_test_split(unmon.record_id, test_size=TEST_SIZE, random_state=SEED)
    split = pd.concat([pd.DataFrame({"record_id": ids, "split": tag})
                       for ids, tag in [(tr_m,"train"),(te_m,"test"),(tr_u,"train"),(te_u,"test")]], ignore_index=True)
    Path(SPLIT_PATH).parent.mkdir(parents=True, exist_ok=True)
    split.to_csv(SPLIT_PATH, index=False)
    print("공유 분할 저장:", SPLIT_PATH)
if split.record_id.isna().any() or split.record_id.duplicated().any() or not split.split.isin(["train","test"]).all():
    raise ValueError("분할 파일의 ID와 train/test 값을 확인하세요.")
if not all_ids.issubset(set(split.record_id)):
    raise ValueError("공유 분할에 없는 기록이 있습니다. 데이터 버전을 확인하세요.")
SPLIT = split.set_index("record_id")["split"]

def attach_split(df):
    df = df.copy()
    df["split"] = df.record_id.map(SPLIT)
    if df.split.isna().any() or not {"train","test"}.issubset(set(df.split)):
        raise ValueError("공유 분할 적용 실패 또는 학습/평가 기록 누락")
    return df

mon, unmon = attach_split(mon), attach_split(unmon)
# 조건별 파일의 행 순서가 달라도 동일한 순서로 정렬해 CV fold까지 맞춥니다.
REFERENCE = {"mon": mon, UNMON_NAME: unmon}

def comparable_data(kind, trim, points):
    if trim == TRIM and points == BASE_POINTS:
        return REFERENCE[kind].copy(), FEATS.copy()
    df, feats = load_features(kind, trim, points)
    ref = REFERENCE[kind]
    if set(df.record_id) != set(ref.record_id):
        raise ValueError(f"{kind}/trim{trim}/{points}지점의 기록 집합이 기준 조건과 다릅니다.")
    df = df.set_index("record_id").loc[ref.record_id].reset_index()
    for col in ["dataset", "page_id", "sample_idx", "label"]:
        if not df[col].reset_index(drop=True).equals(ref[col].reset_index(drop=True)):
            raise ValueError(f"동일 record_id의 {col} 값이 달라요: {kind}/{points}/trim{trim}")
    return attach_split(df), feats

# 지점 수/trim별 메타데이터 차이를 학습 전에 확인합니다.
for p in CUMUL_POINTS_TO_COMPARE:
    for t in sorted(set([TRIM] + TRIMS_TO_COMPARE)):
        m, mf = comparable_data("mon", t, p)
        u, uf = comparable_data(UNMON_NAME, t, p)
        if mf != uf:
            raise ValueError("mon/unmon 특징 열 순서가 다릅니다.")
        del m, u
print(pd.crosstab(pd.concat([mon,unmon],ignore_index=True).dataset, pd.concat([mon,unmon],ignore_index=True).split))

기존 공유 분할 사용: /content/drive/MyDrive/MLB-Project/results/split_seed42.csv
split     test  train
dataset              
mon       3800  15200
unmon10k  2000   8000


## 3. 재개 가능한 그리드 서치·교차검증·평가 공통 함수

5개 후보를 같은 층화 5-fold로 비교하고, 평균 검증 정확도가 가장 높은 후보를 선택해 전체 학습 데이터로 재학습합니다.
ParameterGrid를 직접 순회해 완료한 fold 점수를 즉시 저장합니다. GridSearchCV와 동일한 분할·평가 원리입니다.
후보 하나의 5-fold가 끝날 때만 점수를 출력하고, 긴 작업은 5분마다 실제 완료한 fold 수를 표시합니다.
테스트 기록은 후보 선택에 사용하지 않습니다. CSV에 후보별 점수·표준편차·시간을 저장합니다.

In [5]:
RESULTS, CV_TABLES, PREDS, BEST_PARAMS, FEATURE_SCORES = {}, {}, {}, {}, {}

def make_model(class_weight=None):
    return RandomForestClassifier(random_state=SEED,n_jobs=N_JOBS,class_weight=class_weight)

SEARCH_GRID = RF_PARAM_GRID

def default_grid():
    return {"n_estimators":[100],"max_depth":[None],"max_features":["sqrt"],"min_samples_leaf":[1]}

def fixed_grid(params):
    return {key: [value] for key, value in params.items()}

def save_frame(df, filename, index=False):
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].map(lambda v: json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v)
    out.to_csv(Path(RESULT_DIR)/filename,index=index)

def atomic_json(path, payload):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload,ensure_ascii=False,allow_nan=False),encoding="utf-8")
    os.replace(tmp,path)

def fit_model(X, y, key, points, trim, grid=None, class_weight=None):
    # ParameterGrid의 후보를 직접 순회하는 그리드 서치입니다. GridSearchCV와 같은 층화 fold를 사용합니다.
    candidates = list(ParameterGrid(SEARCH_GRID if grid is None else grid))
    cv = StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    if y.value_counts().min()<N_FOLDS:
        raise ValueError("클래스별 학습 표본 수가 N_FOLDS보다 작습니다.")
    if not candidates or LOG_INTERVAL_SECONDS<=0:
        raise ValueError("후보와 로그 간격을 확인하세요.")
    folds = list(cv.split(X,y))
    signature = joblib_hash(("rf_fold_v1",X,y,N_FOLDS,SEED,sklearn.__version__,make_model(class_weight).get_params()))
    fold_dir = Path(RESULT_DIR)/"completed_folds"/signature
    total = len(candidates)*N_FOLDS
    state = {"done":0,"reused":0,"stage":"CV"}
    t0 = time.perf_counter()
    stop = threading.Event()
    def status():
        while not stop.wait(LOG_INTERVAL_SECONDS):
            print(f"[진행] {key}: {state['done']}/{total} fold 완료 | {state['stage']} | 경과 {(time.perf_counter()-t0)/60:.1f}분",flush=True)
    print(f"[시작] {key}: {len(candidates)}후보 × {N_FOLDS}-fold = {total}회 + 최적 모델 재학습",flush=True)
    reporter = threading.Thread(target=status,daemon=True)
    reporter.start()
    rows=[]
    try:
        for ci,params in enumerate(candidates,1):
            records=[]
            for fi,(itr,ival) in enumerate(folds):
                identity=joblib_hash((signature,params,fi))
                file=fold_dir/f"{identity}.json"
                if USE_FOLD_CACHE and file.is_file():
                    rec=json.loads(file.read_text(encoding="utf-8"))
                    if rec.get("identity")!=identity:
                        raise ValueError("저장된 fold의 식별값이 맞지 않습니다.")
                    state["reused"]+=1
                else:
                    model=make_model(class_weight).set_params(**params)
                    begin=time.perf_counter()
                    model.fit(X.iloc[itr],y.iloc[itr])
                    fit_seconds=time.perf_counter()-begin
                    begin=time.perf_counter()
                    score=float(accuracy_score(y.iloc[ival],model.predict(X.iloc[ival])))
                    score_seconds=time.perf_counter()-begin
                    rec=dict(identity=identity,score=score,fit_time=fit_seconds,score_time=score_seconds)
                    del model
                    if USE_FOLD_CACHE:atomic_json(file,rec)
                records.append(rec)
                state["done"]+=1
            scores=[r["score"] for r in records]
            rows.append({**{f"param_{k}":v for k,v in params.items()},
                         "cv_acc_mean":float(np.mean(scores)),"cv_acc_std":float(np.std(scores)),
                         "mean_fit_time":float(np.mean([r["fit_time"] for r in records])),
                         "mean_score_time":float(np.mean([r["score_time"] for r in records]))})
            print(f"[후보 {ci}/{len(candidates)}] CV={np.mean(scores):.4f} | {state['done']}/{total} fold 완료",flush=True)
        best_index=int(np.argmax([r["cv_acc_mean"] for r in rows]))
        best_params=candidates[best_index].copy()
        state["stage"]="최적 모델 재학습"
        print(f"[재학습] 최적 설정 {best_params}",flush=True)
        best_model=make_model(class_weight).set_params(**best_params)
        refit_start=time.perf_counter()
        best_model.fit(X,y)
        refit_seconds=time.perf_counter()-refit_start
    except BaseException as exc:
        print(f"[중단/오류] {type(exc).__name__}: {exc} (성공적으로 완료한 fold는 저장됩니다)",flush=True)
        raise
    finally:
        stop.set()
        reporter.join(timeout=1)
    elapsed=time.perf_counter()-t0
    table=pd.DataFrame(rows)
    table["rank_test_score"]=table.cv_acc_mean.rank(ascending=False,method="min").astype(int)
    table.insert(0,"experiment",key)
    table["cumul_points"],table["trim"]=points,trim
    CV_TABLES[key]=table
    info=dict(best_params=best_params,cv_acc=rows[best_index]["cv_acc_mean"],cv_std=rows[best_index]["cv_acc_std"],
              search_time_s=elapsed,refit_time_s=refit_seconds,cumul_points=points,n_features=X.shape[1],
              reused_cv_folds=state["reused"])
    print(f"[완료] {key}: CV={info['cv_acc']:.4f} | {elapsed/60:.1f}분 | 재사용 fold {state['reused']}/{total}",flush=True)
    if SHOW_CV_TABLE:display(table.round(4))
    return best_model,info

def store_result(key,row,pred_frame):
    RESULTS[key]=row
    PREDS[key]=pred_frame
    save_frame(pd.DataFrame(RESULTS.values()),"rf_summary_partial.csv")
    save_frame(pd.concat(CV_TABLES.values(),ignore_index=True),"rf_cv_tables_partial.csv")
    return row

In [6]:
def openworld_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[-1,1]).ravel()
    def ratio(a,b): return float(a/b) if b else 0.0
    return dict(mon_as_mon=ratio(tp,tp+fn), unmon_as_unmon=ratio(tn,tn+fp),
                precision=ratio(tp,tp+fp), fpr=ratio(fp,fp+tn), TP=int(tp), FN=int(fn), TN=int(tn), FP=int(fp))

def capture_features(model, task, points, feats):
    if task not in ("closed_world","open_binary"):
        return
    FEATURE_SCORES[(task,points)] = pd.Series(model.feature_importances_,index=feats)

def run_task(task, points, trim=TRIM, experiment=None, grid=None, class_weight=None):
    m, feats = comparable_data("mon", trim, points)
    if task == "closed_world":
        df = m.copy()
        df["y"] = df.label
    else:
        u, uf = comparable_data(UNMON_NAME, trim, points)
        assert feats == uf
        df = pd.concat([m,u], ignore_index=True)
        df["y"] = np.where(df.dataset.eq("mon"),1,-1) if task == "open_binary" else df.label
    tr, te = df[df.split.eq("train")], df[df.split.eq("test")]
    exp = experiment or task
    key = f"{exp}_cumul{points}_trim{trim}"
    model, info = fit_model(tr[feats], tr.y, key, points, trim, grid=grid, class_weight=class_weight)
    t0 = time.perf_counter()
    pred = model.predict(te[feats])
    pred_time = time.perf_counter() - t0
    row = dict(experiment=exp, task=task, trim=trim, **info,
               test_acc=float(accuracy_score(te.y,pred)), predict_time_s=pred_time)
    pf = te[["record_id","dataset","label","y"]].copy().assign(pred=pred)
    if task == "open_binary":
        row.update(openworld_metrics(te.y,pred))
        t0 = time.perf_counter()
        p_mon = model.predict_proba(te[feats])[:, list(model.classes_).index(1)]
        row["proba_time_s"] = time.perf_counter()-t0
        row.update(majority_baseline=float(te.y.value_counts(normalize=True).max()),
                   roc_auc=float(roc_auc_score(te.y,p_mon)),
                   pr_auc=float(average_precision_score(te.y,p_mon,pos_label=1)))
        pf["p_mon"] = p_mon
    else:
        row["test_macro_f1"] = float(f1_score(te.y,pred,average="macro",zero_division=0))
        if task == "open_multi":
            y = te.y.to_numpy()
            row.update(openworld_metrics(np.where(y>=0,1,-1), np.where(pred>=0,1,-1)))
            row["mon_site_exact"] = float((pred[y>=0] == y[y>=0]).mean())
    if exp == task and trim == TRIM:
        BEST_PARAMS[(task,points)] = info["best_params"].copy()
        capture_features(model,task,points,feats)
    del model
    print(f"→ test accuracy: {row['test_acc']:.4f}", flush=True)
    return store_result(key,row,pf)

# 모델 전체를 저장하는 대신 완료한 실험의 결과·예측·중요도·CV 표를 저장합니다.
# 중간에 중단한 GridSearchCV의 fold를 이어받는 기능은 아닙니다.
_run_task_uncached = run_task

def run_task(task, points, trim=TRIM, experiment=None, grid=None, class_weight=None):
    exp = experiment or task
    key = f"{exp}_cumul{points}_trim{trim}"
    m, feats = comparable_data("mon",trim,points)
    if task == "closed_world":
        data = m.copy()
        data["y"] = data.label
    else:
        u, uf = comparable_data(UNMON_NAME,trim,points)
        assert feats == uf
        data = pd.concat([m,u],ignore_index=True)
        data["y"] = np.where(data.dataset.eq("mon"),1,-1) if task == "open_binary" else data.label
        del u
    # 실제 입력·정답·분할·후보·fold·seed·라이브러리 버전이 바뀌면 이전 캐시를 사용하지 않습니다.
    signature = joblib_hash(("rf_completed_v2", key, data[["record_id","split","y"]+feats],
                            SEARCH_GRID if grid is None else grid, class_weight,
                            N_FOLDS, SEED, sklearn.__version__, make_model(class_weight).get_params()))
    del m, data
    cache_dir = Path(RESULT_DIR)/"completed_experiments"
    cache_dir.mkdir(parents=True,exist_ok=True)
    cache_file = cache_dir/f"{key}_{signature}.json"
    if USE_COMPLETED_CACHE and cache_file.is_file():
        with cache_file.open(encoding="utf-8") as f:
            saved = json.load(f)
        if saved.get("signature") != signature:
            raise ValueError("저장된 실험의 식별값이 맞지 않습니다.")
        row = saved["row"]
        pred_frame = pd.DataFrame(saved["predictions"])
        CV_TABLES[key] = pd.DataFrame(saved["cv_table"])
        if exp == task and trim == TRIM:
            BEST_PARAMS[(task,points)] = row["best_params"].copy()
            if saved["feature_scores"] is not None:
                fs = saved["feature_scores"]
                FEATURE_SCORES[(task,points)] = pd.Series(fs["values"],index=fs["names"])
        print(f"[완료 결과 재사용] {key} — 다시 학습하지 않습니다.",flush=True)
        return store_result(key,row,pred_frame)
    row = _run_task_uncached(task,points,trim=trim,experiment=experiment,grid=grid,class_weight=class_weight)
    if USE_COMPLETED_CACHE:
        scores = FEATURE_SCORES.get((task,points)) if exp == task and trim == TRIM else None
        payload = dict(signature=signature,row=row,
                       predictions=PREDS[key].to_dict(orient="records"),
                       cv_table=CV_TABLES[key].to_dict(orient="records"),
                       feature_scores=None if scores is None else dict(names=scores.index.tolist(),values=scores.tolist()))
        temporary = cache_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(payload,ensure_ascii=False,allow_nan=False),encoding="utf-8")
        os.replace(temporary,cache_file)
    return row

## 4. Closed-world: 관심 사이트 식별 + 튜닝 전후 비교

관심 사이트 95개 중 사이트 번호를 맞힙니다. 튜닝한 RF와 기본 설정 RF(트리 100개, 깊이 제한 없음)를 비교합니다. 정확도와 Macro-F1을 기록합니다.

In [7]:
for points in CUMUL_POINTS_TO_COMPARE:
    run_task("closed_world",points)
    run_task("closed_world",points,experiment="closed_world_default",grid=default_grid())

[시작] closed_world_cumul50_trim0: 5후보 × 5-fold = 25회 + 최적 모델 재학습
[후보 1/5] CV=0.6691 | 5/25 fold 완료
[진행] closed_world_cumul50_trim0: 7/25 fold 완료 | CV | 경과 5.0분
[후보 2/5] CV=0.6760 | 10/25 fold 완료
[후보 3/5] CV=0.8476 | 15/25 fold 완료
[진행] closed_world_cumul50_trim0: 15/25 fold 완료 | CV | 경과 10.0분
[진행] closed_world_cumul50_trim0: 18/25 fold 완료 | CV | 경과 15.0분
[후보 4/5] CV=0.6014 | 20/25 fold 완료
[후보 5/5] CV=0.6669 | 25/25 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] closed_world_cumul50_trim0: CV=0.8476 | 18.3분 | 재사용 fold 0/25
→ test accuracy: 0.8561
[시작] closed_world_default_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8518 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] closed_world_default_cumul50_trim0: CV=0.8518 | 2.5분 | 재사용 fold 0/5
→ test accuracy: 0.8624


## 5. Open-world 이진: 관심 사이트인지 판단

mon=1, unmon=-1. 관심 기록 탐지율(TPR), 관심 밖 정답률(TNR), precision, FPR, ROC-AUC, PR-AUC를 기록합니다. mon 확률도 저장합니다.

In [8]:
for points in CUMUL_POINTS_TO_COMPARE:
    run_task("open_binary",points)

[시작] open_binary_cumul50_trim0: 5후보 × 5-fold = 25회 + 최적 모델 재학습
[후보 1/5] CV=0.8324 | 5/25 fold 완료
[진행] open_binary_cumul50_trim0: 8/25 fold 완료 | CV | 경과 5.0분
[후보 2/5] CV=0.8321 | 10/25 fold 완료
[후보 3/5] CV=0.8889 | 15/25 fold 완료
[진행] open_binary_cumul50_trim0: 16/25 fold 완료 | CV | 경과 10.0분
[후보 4/5] CV=0.8546 | 20/25 fold 완료
[진행] open_binary_cumul50_trim0: 21/25 fold 완료 | CV | 경과 15.0분
[후보 5/5] CV=0.8316 | 25/25 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] open_binary_cumul50_trim0: CV=0.8889 | 16.1분 | 재사용 fold 0/25
→ test accuracy: 0.8938


### 5-추가. 균형 가중치 비교

가중치 효과를 비교하기 위해 해당 지점 수의 기본 이진 모델에서 선택한 하이퍼파라미터를 고정하고 class_weight="balanced"만 바꿉니다. balanced 조건에서 별도로 튜닝하지 않습니다.

In [9]:
for points in CUMUL_POINTS_TO_COMPARE:
    run_task("open_binary",points,experiment="open_binary_balanced",class_weight="balanced",
             grid=fixed_grid(BEST_PARAMS[("open_binary",points)]))

[시작] open_binary_balanced_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8893 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] open_binary_balanced_cumul50_trim0: CV=0.8893 | 2.1분 | 재사용 fold 0/5
→ test accuracy: 0.8997


## 6. Open-world 다중: 관심 사이트 번호 + 관심 밖

0~94 및 -1의 96클래스 분류입니다. 사이트 번호까지 맞힌 비율(mon_site_exact)과 관심 여부만 맞힌 비율(mon_as_mon)을 구분합니다.

In [10]:
for points in CUMUL_POINTS_TO_COMPARE:
    run_task("open_multi",points)

[시작] open_multi_cumul50_trim0: 5후보 × 5-fold = 25회 + 최적 모델 재학습
[후보 1/5] CV=0.5622 | 5/25 fold 완료
[진행] open_multi_cumul50_trim0: 6/25 fold 완료 | CV | 경과 5.0분
[진행] open_multi_cumul50_trim0: 9/25 fold 완료 | CV | 경과 10.0분
[후보 2/5] CV=0.5631 | 10/25 fold 완료
[진행] open_multi_cumul50_trim0: 14/25 fold 완료 | CV | 경과 15.0분
[후보 3/5] CV=0.8019 | 15/25 fold 완료
[진행] open_multi_cumul50_trim0: 17/25 fold 완료 | CV | 경과 20.0분
[진행] open_multi_cumul50_trim0: 19/25 fold 완료 | CV | 경과 25.0분
[후보 4/5] CV=0.6025 | 20/25 fold 완료
[후보 5/5] CV=0.5587 | 25/25 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] open_multi_cumul50_trim0: CV=0.8019 | 28.4분 | 재사용 fold 0/25
→ test accuracy: 0.8166


## 7. trim 조건 결과

기본 설정은 trim0만 사용하므로 기존 이진 결과를 재사용합니다. trim0만 실행할 때는 패킷 제거 효과를 비교하지 않습니다. 추가 CSV를 확보하고 `TRIMS_TO_COMPARE`를 확장한 경우에만 다른 trim을 같은 RF 설정으로 다시 학습합니다. 재사용한 행의 시간은 기존 실험 시간입니다.

In [11]:
trim_rows = []
for points in CUMUL_POINTS_TO_COMPARE:
    for trim in TRIMS_TO_COMPARE:
        if trim == TRIM:
            base = RESULTS[f"open_binary_cumul{points}_trim{TRIM}"]
            row = {**base, "experiment":"open_binary_trimcmp", "reused_from":"open_binary"}
            key = f"open_binary_trimcmp_cumul{points}_trim{trim}"
            store_result(key,row,PREDS[f"open_binary_cumul{points}_trim{TRIM}"].copy())
        else:
            row = run_task("open_binary",points,trim=trim,experiment="open_binary_trimcmp",
                           grid=fixed_grid(BEST_PARAMS[("open_binary",points)]))
        trim_rows.append(row)
trim_table = pd.DataFrame(trim_rows)
display(trim_table[["cumul_points","trim","cv_acc","test_acc","mon_as_mon","unmon_as_unmon","roc_auc"]].round(4))
save_frame(trim_table,"rf_trim_comparison.csv")

,cumul_points,trim,cv_acc,test_acc,mon_as_mon,unmon_as_unmon,roc_auc
0,50,0,0.8889,0.8938,0.9639,0.7605,0.9648


### 7-추가. 원본 패킷 방향 보조 분석 (기본 생략)

`RUN_RAW_DIRECTION=False`이므로 원본 pkl 없이 실행합니다. 추후 켜면 위치별 송신 비율을 계산합니다. 이 표만으로 성능의 원인이나 수집 편향 제거를 단정하지 않습니다.

In [12]:
if RUN_RAW_DIRECTION:
    raw_name = {"unmon10k":"unmon_standard10.pkl","unmon3k":"unmon_standard10_3000.pkl"}.get(UNMON_NAME)
    pm = Path(RAW_DIR)/"mon_standard.pkl"
    pu = Path(RAW_DIR)/(raw_name or "unknown.pkl")
    if not pm.is_file() or not pu.is_file():
        print("원본 pkl이 없어 패킷 방향 분석만 생략:", pm, pu)
    else:
        with pm.open("rb") as f: raw_m=pickle.load(f)
        with pu.open("rb") as f: raw_u=pickle.load(f)
        # 원본 노트북의 mon 구조: 페이지 번호 -> 해당 페이지의 접속 기록 목록
        keys = sorted(raw_m) if isinstance(raw_m,dict) else range(len(raw_m))
        records_m = [r for k in keys for r in raw_m[k]]
        def dir_matrix(records,n=40):
            out = np.full((len(records),n),np.nan)
            for i,r in enumerate(records):
                a=np.asarray(r[:n]); out[i,:len(a)]=(a>0).astype(float)
            return out
        dm,du=dir_matrix(records_m),dir_matrix(raw_u)
        directions=pd.DataFrame({"packet_pos":np.arange(40),"mon_out_ratio":np.nanmean(dm,axis=0),
                                 "unmon_out_ratio":np.nanmean(du,axis=0)})
        directions["abs_diff"]=(directions.mon_out_ratio-directions.unmon_out_ratio).abs()
        display(directions.round(3))
        save_frame(directions,"rf_early_packet_direction.csv")
        del raw_m,raw_u,records_m,dm,du

## 8. 특징 해석

feature_importances_는 불순도 감소 기반 중요도입니다. 부호나 mon/unmon 방향을 나타내지 않으며 인과 효과도 아닙니다. 고유값 수와 상관된 특징의 영향을 받습니다. 그룹별 합·평균을 함께 보고 그룹 제거 실험과 비교하세요.

In [13]:
for points in CUMUL_POINTS_TO_COMPARE:
    closed = FEATURE_SCORES[("closed_world",points)]
    binary = FEATURE_SCORES[("open_binary",points)]
    feature_scores = pd.concat([closed.rename("closed_importance"),binary.rename("binary_importance")],axis=1)
    group_scores = []
    for group in ["g1","g2","g3","g4","g5","g6"]:
        cols = [c for c in closed.index if c.startswith(group+"_")]
        group_scores.append(dict(group=group,n_features=len(cols),
                                 closed_sum=float(closed[cols].sum()),closed_mean=float(closed[cols].mean()),
                                 binary_sum=float(binary[cols].sum()),binary_mean=float(binary[cols].mean())))
    print(f"[{points} 지점] 특징 중요도 / 계수 크기 (상위 15개)")
    display(feature_scores.sort_values("closed_importance",ascending=False).head(15).round(4))
    display(pd.DataFrame(group_scores).round(4))
    save_frame(feature_scores,"rf_feature_importance_cumul"+str(points)+".csv",index=True)
    save_frame(pd.DataFrame(group_scores),"rf_group_importance_cumul"+str(points)+".csv")
    pass
    m, feats = comparable_data("mon",TRIM,points)
    u, _ = comparable_data(UNMON_NAME,TRIM,points)
    joined = pd.concat([m[feats],u[feats]],ignore_index=True)
    print("전체 데이터에서 상수인 특징:", joined.columns[joined.nunique()<=1].tolist())
    del joined,m,u

[50 지점] 특징 중요도 / 계수 크기 (상위 15개)


,closed_importance,binary_importance
g4_first100_out_ratio,0.0292,0.0430
g4_first30_out_ratio,0.0243,0.1032
g6_cumul_01,0.0222,0.0344
g1_n_out,0.0212,0.0095
g1_in_ratio,0.0199,0.0105
g2_chunk20_mean,0.0195,0.0101
g2_chunk100_mean,0.0195,0.0095
g2_chunk50_mean,0.0189,0.0099
g6_cumul_02,0.0184,0.0205
g2_out_pos_mean,0.0174,0.0089


,group,n_features,closed_sum,closed_mean,binary_sum,binary_mean
0,g1,3,0.0550,0.0183,0.0285,0.0095
1,g2,17,0.1715,0.0101,0.1277,0.0075
2,g3,6,0.0677,0.0113,0.0706,0.0118
3,g4,3,0.0669,0.0223,0.1561,0.0520
4,g5,31,0.2023,0.0065,0.2842,0.0092
5,g6,50,0.4366,0.0087,0.3329,0.0067


전체 데이터에서 상수인 특징: ['g6_cumul_00']


### 8-추가. 사이트별 정확도와 주요 혼동 쌍

In [14]:
for points in CUMUL_POINTS_TO_COMPARE:
    pc = PREDS[f"closed_world_cumul{points}_trim{TRIM}"]
    site_acc = pc.assign(correct=pc.y.eq(pc.pred)).groupby("y").correct.mean().rename("test_acc").reset_index()
    wrong = pc[pc.y.ne(pc.pred)]
    pairs = wrong.groupby(["y","pred"]).size().sort_values(ascending=False).head(15).rename("count").reset_index()
    print(f"[{points} 지점] 사이트별 낮은 정확도 / 주요 혼동 쌍")
    display(site_acc.sort_values("test_acc").head(10))
    display(pairs)
    save_frame(site_acc,f"rf_closed_site_acc_cumul{points}.csv")
    save_frame(pairs,f"rf_closed_confused_pairs_cumul{points}.csv")

[50 지점] 사이트별 낮은 정확도 / 주요 혼동 쌍


,y,test_acc
24,24,0.575
14,14,0.625
13,13,0.650
47,47,0.650
55,55,0.650
16,16,0.700
22,22,0.700
11,11,0.725
34,34,0.725
65,65,0.725


,y,pred,count
0,32,89,6
1,14,51,5
2,0,7,3
3,34,54,3
4,16,74,3
5,22,45,3
6,10,58,3
7,47,91,3
8,60,72,2
9,24,10,2


## 9. 특징 그룹 제거 실험

Closed-world와 Open-world 이진 모두 실행합니다. g1~g6를 하나씩 제거하고 기본 모델에서 선택한 설정을 고정한 채 5-fold 평균 정확도를 비교합니다.
전체 특징 점수는 기본 실험에서 재사용하고, 제거 조건마다 새로 학습합니다. 완료한 fold는 중단 후 재사용할 수 있습니다.
음수 delta는 제거 후 검증 정확도가 내려간 것입니다. 테스트 점수로 그룹을 선택하지 않습니다.
`RUN_ABLATION=True`가 기본이므로 전체 실행에 포함됩니다.

In [15]:
if RUN_ABLATION:
    for points in CUMUL_POINTS_TO_COMPARE:
        m, feats = comparable_data("mon",TRIM,points)
        u, uf = comparable_data(UNMON_NAME,TRIM,points)
        for task in ["closed_world","open_binary"]:
            df = m.copy() if task == "closed_world" else pd.concat([m,u],ignore_index=True)
            df["y"] = df.label if task == "closed_world" else np.where(df.dataset.eq("mon"),1,-1)
            tr = df[df.split.eq("train")]
            rows = []
            for group in [None,"g1","g2","g3","g4","g5","g6"]:
                remaining = feats if group is None else [c for c in feats if not c.startswith(group+"_")]
                if group is None:
                    info = RESULTS[f"{task}_cumul{points}_trim{TRIM}"]
                else:
                    key = f"ablation_{task}_minus_{group}_cumul{points}_trim{TRIM}"
                    fitted,info = fit_model(tr[remaining],tr.y,key,points,TRIM,
                                           grid=fixed_grid(BEST_PARAMS[(task,points)]))
                    del fitted
                rows.append(dict(removed=group or "none",n_features=len(remaining),cumul_points=points,
                                 best_params=info["best_params"],cv_acc=info["cv_acc"],cv_std=info["cv_std"]))
            abl = pd.DataFrame(rows)
            abl["delta_vs_full"] = abl.cv_acc-abl.cv_acc.iloc[0]
            display(abl.round(4))
            save_frame(abl,f"rf_ablation_{task}_cumul{points}.csv")
else:
    print("RUN_ABLATION=False → 양쪽 분류의 그룹 제거 실험 생략")

[시작] ablation_closed_world_minus_g1_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8462 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] ablation_closed_world_minus_g1_cumul50_trim0: CV=0.8462 | 2.4분 | 재사용 fold 0/5
[시작] ablation_closed_world_minus_g2_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8428 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] ablation_closed_world_minus_g2_cumul50_trim0: CV=0.8428 | 2.3분 | 재사용 fold 0/5
[시작] ablation_closed_world_minus_g3_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8464 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] ablation_closed_world_minus_g3_cumul50_trim0: CV=0.8464 | 2.4분 | 재사용 fold 0/5
[시작] ablation_closed_world_minus_g4_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8213 | 5/5 fold 완료
[

,removed,n_features,cumul_points,best_params,cv_acc,cv_std,delta_vs_full
0,none,110,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8476,0.0068,0.0000
1,g1,107,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8462,0.0091,-0.0014
2,g2,93,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8428,0.0053,-0.0049
3,g3,104,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8464,0.0061,-0.0012
4,g4,107,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8212,0.0086,-0.0264
5,g5,79,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8366,0.0044,-0.0111
6,g6,60,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8258,0.0081,-0.0218


[시작] ablation_open_binary_minus_g1_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8878 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] ablation_open_binary_minus_g1_cumul50_trim0: CV=0.8878 | 2.2분 | 재사용 fold 0/5
[시작] ablation_open_binary_minus_g2_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8861 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] ablation_open_binary_minus_g2_cumul50_trim0: CV=0.8861 | 2.1분 | 재사용 fold 0/5
[시작] ablation_open_binary_minus_g3_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8844 | 5/5 fold 완료
[재학습] 최적 설정 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
[완료] ablation_open_binary_minus_g3_cumul50_trim0: CV=0.8844 | 2.2분 | 재사용 fold 0/5
[시작] ablation_open_binary_minus_g4_cumul50_trim0: 1후보 × 5-fold = 5회 + 최적 모델 재학습
[후보 1/1] CV=0.8541 | 5/5 fold 완료
[재학습] 최적

,removed,n_features,cumul_points,best_params,cv_acc,cv_std,delta_vs_full
0,none,110,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8889,0.0041,0.0000
1,g1,107,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8878,0.0019,-0.0011
2,g2,93,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8861,0.0026,-0.0028
3,g3,104,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8844,0.0034,-0.0045
4,g4,107,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8541,0.0022,-0.0349
5,g5,79,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8812,0.0035,-0.0078
6,g6,60,50,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8810,0.0039,-0.0079


## 10. 실험 요약 및 전체 결과 저장

세 기본 분류와 추가 실험 결과를 CSV로 저장합니다. 누적 흐름은 모두 50개 지점입니다. `rf_main_results.csv`는 세 기본 분류의 요약이고 `rf_summary.csv`는 추가 실험을 포함합니다. 매 실험 후 `rf_summary_partial.csv`도 갱신합니다.

In [16]:
summary = pd.DataFrame(RESULTS.values())
if summary.empty:
    raise RuntimeError("학습 결과가 없습니다. 4번부터 실행하세요.")
comparison=summary[summary.experiment.isin(["closed_world","open_binary","open_multi"])].copy()
comparison=comparison.sort_values(["task","cumul_points"])
display(comparison[["task","cumul_points","n_features","best_params","cv_acc","cv_std","test_acc","search_time_s"]].round(4))
save_frame(summary,"rf_summary.csv")
save_frame(comparison,"rf_main_results.csv")
save_frame(pd.concat(CV_TABLES.values(),ignore_index=True),"rf_cv_tables.csv")
for key,pf in PREDS.items():
    save_frame(pf,f"rf_pred_{key}.csv")
save_frame(pd.DataFrame(MANIFEST),"rf_input_manifest.csv")
config={"model":"random_forest","seed":SEED,"split_path":SPLIT_PATH,"cumul_points":CUMUL_POINTS_TO_COMPARE,
        "trim":TRIM,"trims":TRIMS_TO_COMPARE,"n_folds":N_FOLDS,"search_grid":SEARCH_GRID,
        "run_ablation":RUN_ABLATION,"selection_metric":"CV accuracy; test is never used for tuning"}
with open(Path(RESULT_DIR)/"rf_run_config.json","w",encoding="utf-8") as f:
    json.dump(config,f,ensure_ascii=False,indent=2)
print("저장 완료:",RESULT_DIR)
print("기본 분류 요약:",f"rf_main_results.csv")

,task,cumul_points,n_features,best_params,cv_acc,cv_std,test_acc,search_time_s
0,closed_world,50,110,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8476,0.0068,0.8561,1095.5484
2,open_binary,50,110,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8889,0.0041,0.8938,966.9215
4,open_multi,50,110,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",0.8019,0.0060,0.8166,1702.5401


저장 완료: /content/drive/MyDrive/MLB-Project/results/random_forest_50points_required
기본 분류 요약: rf_main_results.csv
